# Prepare Training Data — AI Therapist

Downloads Hugging Face psychology datasets, converts them to **Qwen chat format**, and writes merged files to Google Drive.

## Sources
- **SFT:** `LuangMV97/Empathetic_counseling_Dataset`, `UKPLab/Graph2Counsel`, `psychologie-et-serenite/articles-metadata` (+ public API)
- **Eval only:** `AI-companionship/INTIMA` (prompts without labels)

## Outputs (on Drive)
- `clinical_synthetic_data.json` — merged training rows
- `intima_eval.json` — evaluation prompts
- `dataset_stats.json` — row counts per source

**Runtime:** CPU is fine. Article fetching takes ~10–15 min (923 articles @ 0.5s each).

## 1. Install dependencies

In [ ]:
!pip install -q "datasets>=2.19.0" "transformers>=4.44.0" "requests>=2.31.0"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/ai-therapist"
!mkdir -p "{PROJECT_DIR}"
print(f"Output dir: {PROJECT_DIR}")

## 3. Get repo scripts (auto-clone)

Set `REPO_URL` to your GitHub repo (public or private with token). The notebook clones into `/content/therapist-model` and imports `scripts/prepare_datasets.py`.

**First time:** push this repo to GitHub, then paste the clone URL below.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# --- Set your repo URL (HTTPS). For private repos, use a token in Colab Secrets. ---
REPO_URL = "https://github.com/YOUR_USERNAME/Therapist-model.git"  # <-- change this

# Optional: read URL from Colab Secrets instead of editing above
try:
    from google.colab import userdata
    REPO_URL = userdata.get("THERAPIST_REPO_URL")
except Exception:
    pass

CLONE_DIR = Path("/content/therapist-model")
SCRIPT_PATH = CLONE_DIR / "scripts" / "prepare_datasets.py"

# Fallback locations if you already have the repo on Drive
DRIVE_REPO_CANDIDATES = [
    Path("/content/drive/MyDrive/Therapist model/scripts/prepare_datasets.py"),
    Path("/content/drive/MyDrive/therapist-model/scripts/prepare_datasets.py"),
]


def clone_or_update_repo(url: str, dest: Path) -> None:
    if dest.exists() and (dest / ".git").exists():
        print(f"Updating existing clone at {dest}...")
        subprocess.run(["git", "-C", str(dest), "pull", "--ff-only"], check=False)
        return
    if dest.exists():
        import shutil
        shutil.rmtree(dest)
    print(f"Cloning {url} -> {dest}")
    subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)


if SCRIPT_PATH.exists():
    print(f"Using existing clone: {SCRIPT_PATH}")
else:
    cloned = False
    for candidate in DRIVE_REPO_CANDIDATES:
        if candidate.exists():
            SCRIPT_PATH = candidate
            cloned = True
            print(f"Using script from Drive: {SCRIPT_PATH}")
            break

    if not cloned:
        if "YOUR_USERNAME" in REPO_URL:
            raise ValueError(
                "Set REPO_URL to your GitHub repo, or add THERAPIST_REPO_URL in Colab Secrets.\n"
                "Example: https://github.com/you/Therapist-model.git"
            )
        clone_or_update_repo(REPO_URL, CLONE_DIR)
        if not SCRIPT_PATH.exists():
            raise FileNotFoundError(f"Clone succeeded but script not found at {SCRIPT_PATH}")

sys.path.insert(0, str(SCRIPT_PATH.parent))
from prepare_datasets import prepare_all

print(f"Ready — using: {SCRIPT_PATH}")

## 4. Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"
MAX_EMPATHETIC = 10_000   # subsample for first Colab run
ARTICLE_MAX_CHARS = 1500
ARTICLE_RATE_LIMIT = 0.5  # seconds between API calls
SKIP_ARTICLES = False     # set True to skip ~10 min of API fetching

## 5. Download, convert, and save

In [ ]:
stats = prepare_all(
    project_dir=PROJECT_DIR,
    model_id=MODEL_ID,
    max_empathetic=MAX_EMPATHETIC,
    article_max_chars=ARTICLE_MAX_CHARS,
    article_rate_limit=ARTICLE_RATE_LIMIT,
    skip_articles=SKIP_ARTICLES,
)

## 6. Preview output

In [ ]:
import json

data_path = f"{PROJECT_DIR}/clinical_synthetic_data.json"
with open(data_path, encoding="utf-8") as f:
    rows = json.load(f)

print(f"Total SFT rows: {len(rows)}")
print("\n--- First example (truncated) ---")
print(rows[0]["text"][:600], "...")

---

**Next:** Open [`ai_therapist_qlora_training.ipynb`](./ai_therapist_qlora_training.ipynb), set `USE_SAMPLE_DATA = False`, and run training on the prepared JSON.